In [ ]:
"""
All of Us - CMT Phenotype + Principal Component Extraction Script

This script extracts and merges:
  1. All srWGS participants
  2. CMT-related phenotypes (0/1 per person)
  3. Demographic covariates (age, sex, ancestry)
  4. First 10 genetic principal components (PC1-PC10)

Output: one row per person_id for all srWGS participants.
"""

import pandas as pd
import ast
import subprocess
import os
from google.cloud import bigquery

client = bigquery.Client()
DATASET = os.environ["WORKSPACE_CDR"]
print(f"Using dataset: {DATASET}")


# SECTION 1: ALL PARTICIPANTS WITH srWGS DATA


all_participants_query = f"""
SELECT DISTINCT
    p.person_id
FROM
    `{DATASET}.person` p
INNER JOIN
    `{DATASET}.cb_search_person` csp ON p.person_id = csp.person_id
WHERE
    csp.has_whole_genome_variant = 1
"""

print("\nExtracting all participants with srWGS data...")
all_participants = client.query(all_participants_query).to_dataframe()
print(f"  Total participants with srWGS: {len(all_participants)}")


# SECTION 2: CMT-RELATED PHENOTYPES
# OMOP concept IDs from Supplementary Table 7


phenotype_query = f"""
SELECT DISTINCT
    co.person_id,
    CASE
        WHEN co.condition_concept_id = 436799  THEN 'pes_cavus'
        WHEN co.condition_concept_id = 433577  THEN 'hammer_toes'
        WHEN co.condition_concept_id = 4264617 THEN 'foot_drop'
        WHEN co.condition_concept_id = 437584  THEN 'ataxia'
        WHEN co.condition_concept_id = 79908   THEN 'muscle_weakness'
        WHEN co.condition_concept_id = 4236484 THEN 'paraesthesia'
        WHEN co.condition_concept_id = 436096  THEN 'chronic_pain'
        WHEN co.condition_concept_id = 373852  THEN 'neuralgia'
    END AS phenotype,
    1 AS present
FROM
    `{DATASET}.condition_occurrence` co
WHERE
    co.condition_concept_id IN (
        436799,   -- Pes cavus        (SNOMED 205091006)
        433577,   -- Hammer toes      (SNOMED 122481008)
        4264617,  -- Foot drop        (SNOMED 6077001)
        437584,   -- Ataxia           (SNOMED 20262006)
        79908,    -- Muscle weakness  (SNOMED 26544005)
        4236484,  -- Paraesthesia     (SNOMED 91019004)
        436096,   -- Chronic pain     (SNOMED 82423001)
        373852    -- Neuralgia        (SNOMED 16269008)
    )
"""

print("\nExtracting CMT-related phenotypes...")
phenotypes = client.query(phenotype_query).to_dataframe()

# Pivot to wide format: one row per person, one column per phenotype
phenotypes_wide = phenotypes.pivot_table(
    index='person_id',
    columns='phenotype',
    values='present',
    aggfunc='max',
    fill_value=0
).reset_index()

# Ensure all expected columns present even if no cases found
expected_cols = [
    'pes_cavus', 'hammer_toes', 'foot_drop', 'ataxia',
    'muscle_weakness', 'paraesthesia', 'chronic_pain', 'neuralgia'
]
for col in expected_cols:
    if col not in phenotypes_wide.columns:
        phenotypes_wide[col] = 0

phenotypes_wide = phenotypes_wide[['person_id'] + expected_cols]

print(f"  Individuals with at least one phenotype: {len(phenotypes_wide)}")
print(f"\n  Phenotype counts (among those with data):")
for col in expected_cols:
    print(f"    {col}: {phenotypes_wide[col].sum()}")


# SECTION 3: DEMOGRAPHIC DATA
# Age, sex, ancestry - covariates for regression models

demographics_query = f"""
SELECT
    p.person_id,
    gc.concept_name                                         AS sex,
    EXTRACT(YEAR FROM CURRENT_DATE()) - p.year_of_birth    AS age,
    rc.concept_name                                         AS ancestry
FROM
    `{DATASET}.person` p
LEFT JOIN
    `{DATASET}.concept` gc ON p.gender_concept_id = gc.concept_id
LEFT JOIN
    `{DATASET}.concept` rc ON p.race_concept_id = rc.concept_id
"""

print("\nExtracting demographic data...")
demographics = client.query(demographics_query).to_dataframe()
print(f"  Demographics extracted for {len(demographics)} individuals")


# SECTION 4: PRINCIPAL COMPONENTS (PC1-PC10)
# Copied from AoU controlled tier ancestry file and parsed
# locally to avoid requiring a Dataproc/Hail runtime

ancestry_path = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv"

print("\nCopying ancestry file from controlled tier...")
subprocess.run([
    'gsutil', '-u', os.environ['GOOGLE_PROJECT'],
    'cp', ancestry_path, 'ancestry_preds.tsv'
], check=True)

print("Parsing principal components...")
ancestry = pd.read_csv('ancestry_preds.tsv', sep='\t')
print(f"  Loaded {len(ancestry)} individuals from ancestry file")

# Parse pca_features from string to list
ancestry['pca_features'] = ancestry['pca_features'].apply(ast.literal_eval)

# Expand into PC columns, keep first 10 only
pc_cols = [f'PC{i+1}' for i in range(10)]
pc_df = pd.DataFrame(
    ancestry['pca_features'].tolist(),
    columns=[f'PC{i+1}' for i in range(16)]
)[pc_cols]

# Combine with person_id
pcs = pd.concat([
    ancestry[['research_id']].rename(columns={'research_id': 'person_id'}),
    pc_df
], axis=1)

# Sanity check
missing_pcs = pcs[pc_cols].isnull().sum()
if missing_pcs.sum() == 0:
    print("  No missing values in PC columns")
else:
    print(f"  Warning - missing values in PCs:\n{missing_pcs[missing_pcs > 0]}")

# Clean up local copy of ancestry file
os.remove('ancestry_preds.tsv')
print("  Temporary ancestry file removed")


# SECTION 5: ASSEMBLE FINAL OUTPUT
# Start from ALL srWGS participants, left join everything
# so no participant is dropped

print("\nAssembling final output...")

# Left join phenotypes (non-carriers get 0)
output = all_participants.merge(phenotypes_wide, on='person_id', how='left')
for col in expected_cols:
    output[col] = output[col].fillna(0).astype(int)

# Left join demographics
output = output.merge(demographics, on='person_id', how='left')

# Left join PCs
output = output.merge(pcs, on='person_id', how='left')

print(f"  Final dataset: {len(output)} individuals x {len(output.columns)} columns")
print(f"  Columns: {list(output.columns)}")

# Sanity checks
print(f"\n  Phenotype counts (full cohort):")
for col in expected_cols:
    print(f"    {col}: {output[col].sum()}")

missing_pc_final = output[pc_cols].isnull().sum().sum()
if missing_pc_final > 0:
    print(f"\n  Warning: {missing_pc_final} missing PC values in final output")
    print("  These individuals may not have srWGS ancestry predictions available")
else:
    print(f"\n  All {len(output)} individuals have PC values")


# SECTION 6: EXPORT

output_path = 'pheno_data_phenotypes.csv'
output.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")
print("Join to carrier lists using person_id before running regression models.")

In [ ]:
"""
PMP22 Duplication Carrier Extraction from VCF
==============================================
Loads a VCF containing PMP22 duplication calls,
extracts carriers (any non-reference genotype),
and adds a 'pmp22_duplication' column to the
existing phenotype dataframe.

Adjust VCF_PATH to your actual VCF file location.
"""

import pandas as pd
import gzip

# SECTION 1: LOAD PHENOTYPE DATAFRAME
# If running standalone, load from the previously saved CSV.
# If running as a continuation of cmt_aou_phenotypes_pcs.py,
# comment this section out as 'output' will already be in memory.

phenotype_path = 'pheno_data_phenotypes.csv'
print(f"Loading phenotype dataframe from: {phenotype_path}")
output = pd.read_csv(phenotype_path)
print(f"  Loaded {len(output)} individuals")


# SECTION 2: LOAD VCF AND EXTRACT CARRIERS
# Handles both plain text (.vcf) and gzip compressed (.vcf.gz)

VCF_PATH = 'chr17_dup_filtered.vcf'  # update to your actual VCF path

print(f"\nLoading VCF from: {VCF_PATH}")

def parse_vcf(vcf_path):
    """
    Parse a VCF file and return:
      - sample_ids: list of sample IDs from the header
      - records: list of dicts, one per variant row
    Handles both .vcf and .vcf.gz files.
    """
    sample_ids = []
    records = []

    opener = gzip.open if vcf_path.endswith('.gz') else open
    mode = 'rt'

    with opener(vcf_path, mode) as f:
        for line in f:
            # Skip meta-information lines
            if line.startswith('##'):
                continue
            # Header line contains sample IDs
            if line.startswith('#CHROM'):
                fields = line.strip().split('\t')
                # Sample IDs start after FORMAT column (index 9 onwards)
                sample_ids = fields[9:]
                continue
            # Variant data lines
            fields = line.strip().split('\t')
            chrom   = fields[0]
            pos     = fields[1]
            ref     = fields[3]
            alt     = fields[4]
            fmt     = fields[8]
            samples = fields[9:]
            records.append({
                'CHROM':   chrom,
                'POS':     pos,
                'REF':     ref,
                'ALT':     alt,
                'FORMAT':  fmt,
                'samples': samples
            })

    return sample_ids, records


sample_ids, records = parse_vcf(VCF_PATH)
print(f"  Variants in VCF: {len(records)}")
print(f"  Samples in VCF:  {len(sample_ids)}")


# SECTION 3: IDENTIFY CARRIERS
# A carrier is any individual with a non-reference genotype
# (0/1 heterozygous or 1/1 homozygous) at any variant in the VCF.
#
# To restrict to heterozygous only, change the carrier check to:
#     gt in ('0/1', '0|1')

def get_gt_index(format_str):
    """Return the index of the GT field within the FORMAT string."""
    fields = format_str.split(':')
    return fields.index('GT') if 'GT' in fields else 0


carrier_ids = set()

for record in records:
    gt_index = get_gt_index(record['FORMAT'])
    for sample_id, sample_data in zip(sample_ids, record['samples']):
        gt = sample_data.split(':')[gt_index]
        # Normalise phased (|) and unphased (/) genotypes
        gt_alleles = gt.replace('|', '/').split('/')
        # Carrier = any non-reference, non-missing allele present
        if '.' not in gt_alleles and any(a != '0' for a in gt_alleles):
            carrier_ids.add(sample_id)

print(f"\n  Total carriers identified: {len(carrier_ids)}")

# Summary of genotype types (for QC)
het_count = 0
hom_count = 0
for record in records:
    gt_index = get_gt_index(record['FORMAT'])
    for sample_id, sample_data in zip(sample_ids, record['samples']):
        gt = sample_data.split(':')[gt_index].replace('|', '/')
        if gt in ('0/1', '1/0'):
            het_count += 1
        elif gt == '1/1':
            hom_count += 1

print(f"  Heterozygous calls: {het_count}")
print(f"  Homozygous calls:   {hom_count}")


# SECTION 4: ADD CARRIER COLUMN TO PHENOTYPE DATAFRAME
# person_id in the dataframe is matched against sample IDs in VCF.
# Non-carriers and individuals not in the VCF receive 0.

print("\nAdding pmp22_duplication column to phenotype dataframe...")

# Convert person_id to string for consistent matching
output['person_id'] = output['person_id'].astype(str)
carrier_ids_str = {str(i) for i in carrier_ids}

output['pmp22'] = output['person_id'].isin(carrier_ids_str).astype(int)

# Summary
n_carriers_matched = output['pmp22'].sum()
n_unmatched = len(carrier_ids_str) - n_carriers_matched
print(f"  Carriers matched to phenotype dataframe: {n_carriers_matched}")
if n_unmatched > 0:
    print(f"  Warning: {n_unmatched} VCF carriers not found in phenotype dataframe")
    print("  This may indicate a sample ID format mismatch - check person_id vs VCF sample names")


# SECTION 5: EXPORT UPDATED DATAFRAME

output_path = 'pheno_data.csv'
output.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")
print(f"Final dataframe: {len(output)} individuals x {len(output.columns)} columns")
print(f"Columns: {list(output.columns)}")

# Read and combine chromosome-level VCFs into a single file

In [ ]:
!pip install polars

In [ ]:
import polars as pl
import glob 

def read_vcf(filepath):
    """Read a VCF file, correctly skipping ## meta-lines, using Polars for speed."""
    with open(filepath, 'r') as f:
        skip = sum(1 for line in f if line.startswith('##'))
    df = pl.read_csv(filepath, separator='\t', skip_rows=skip, infer_schema_length=0)
    df.columns = [c.lstrip('#') for c in df.columns]
    return df.to_pandas()


In [ ]:
!gsutil cp -m gs://fc-secure-1ee5bf43-0b6b-4164-a355-ff45dfe2ae3a/data/cmt/counts/gene_regions/variant_extraction/*_full.vcf .

In [ ]:

whole_files = sorted(glob.glob('*_full.vcf'))
print(f'Downloaded {len(whole_files)} whole-population VCF files')
for f in whole_files:
    print(f'  {f}')

# Load and combine all per-chromosome VCFs into a single dataframe
whole_data = [read_vcf(f) for f in whole_files]
whole_vcf  = pd.concat(whole_data, ignore_index=True)

print(f'Total variants across all chromosomes: {len(whole_vcf):,}')
print(f'Unique variant IDs:                    {whole_vcf["ID"].nunique():,}')
print(f'Chromosomes present:                   {sorted(whole_vcf["CHROM"].unique())}')

whole_vcf.to_csv("whole_vcf.vcf", sep = "\t", index = False)